In [ ]:
from __future__ import annotations

import csv
import math
import re
from pathlib import Path

GO_PATTERN = re.compile(r"GO:\d{7}")


In [ ]:
OBO_PATH = Path("/Users/patj/Desktop/pe3/paper/baselines/comparison/data/go-basic.obo")
TEST_TSV_PATH = Path("/Users/patj/Desktop/pe3/paper/baselines/comparison/data/test.tsv")

NAMESPACE_SUFFIX = {
    "molecular_function": "mf",
    "biological_process": "bp",
    "cellular_component": "cc",
}

CATEGORY_ALIASES = {
    "mf": "mf",
    "mm": "mf",
    "bp": "bp",
    "cc": "cc",
}

CATEGORY_TO_NAMESPACE = {
    "mf": "molecular_function",
    "bp": "biological_process",
    "cc": "cellular_component",
}

CATEGORY_TO_TEST_COLUMN = {
    "mf": "Gene Ontology (molecular function)",
    "cc": "Gene Ontology (cellular component)",
    "bp": "Gene Ontology (biological process)",
}

ROOT_GO_BY_CATEGORY = {
    "mf": "GO:0003674",
    "bp": "GO:0008150",
    "cc": "GO:0005575",
}


In [ ]:
def load_go_metadata(obo_path: Path) -> tuple[dict[str, str], dict[str, str], dict[str, list[str]]]:
    namespace_by_go: dict[str, str] = {}
    primary_by_go: dict[str, str] = {}
    parents_by_go: dict[str, list[str]] = {}
    current_id: str | None = None
    current_namespace: str | None = None
    current_alt_ids: list[str] = []
    current_parents: list[str] = []

    def flush_term() -> None:
        nonlocal current_id, current_namespace, current_alt_ids, current_parents
        if current_id and current_namespace:
            namespace_by_go[current_id] = current_namespace
            primary_by_go[current_id] = current_id
            parents_by_go[current_id] = list(current_parents)
            for alt_id in current_alt_ids:
                namespace_by_go[alt_id] = current_namespace
                primary_by_go[alt_id] = current_id
        current_id = None
        current_namespace = None
        current_alt_ids = []
        current_parents = []

    with obo_path.open() as handle:
        for raw_line in handle:
            line = raw_line.rstrip("\n")
            if line == "[Term]":
                flush_term()
                continue
            if not line:
                flush_term()
                continue
            if line.startswith("id: "):
                current_id = line[4:]
            elif line.startswith("namespace: "):
                current_namespace = line[11:]
            elif line.startswith("alt_id: "):
                current_alt_ids.append(line[8:])
            elif line.startswith("is_a: "):
                current_parents.append(line[6:].split(" ! ", 1)[0])

    flush_term()
    return namespace_by_go, primary_by_go, parents_by_go


def split_tsv(tsv_path: Path, namespace_by_go: dict[str, str]) -> dict[str, object]:
    output_paths = {
        namespace: tsv_path.with_name(f"{tsv_path.stem}_{suffix}.tsv")
        for namespace, suffix in NAMESPACE_SUFFIX.items()
    }
    counts = {suffix: 0 for suffix in NAMESPACE_SUFFIX.values()}
    missing_counts: dict[str, int] = {}
    total_rows = 0
    missing_rows = 0
    handles = {namespace: path.open("w") for namespace, path in output_paths.items()}

    try:
        with tsv_path.open() as source:
            for line_no, raw_line in enumerate(source, start=1):
                line = raw_line.rstrip("\n")
                if not line:
                    continue

                parts = line.split("\t")
                if len(parts) < 2:
                    raise ValueError(f"{tsv_path}:{line_no} does not have at least 2 columns")

                go_id = parts[1]
                total_rows += 1
                namespace = namespace_by_go.get(go_id)
                if namespace is None:
                    missing_rows += 1
                    missing_counts[go_id] = missing_counts.get(go_id, 0) + 1
                    continue

                handles[namespace].write(raw_line)
                counts[NAMESPACE_SUFFIX[namespace]] += 1
    finally:
        for handle in handles.values():
            handle.close()

    return {
        "total_rows": total_rows,
        "counts": counts,
        "missing_rows": missing_rows,
        "missing_counts": missing_counts,
        "output_paths": output_paths,
    }


def main(path_to_tsv: str) -> None:
    tsv_path = Path(path_to_tsv)
    namespace_by_go, _, _ = load_go_metadata(OBO_PATH)
    result = split_tsv(tsv_path, namespace_by_go)

    print(f"source_file={tsv_path}")
    print(f"total_rows={result['total_rows']}")
    print(f"mf_rows={result['counts']['mf']}")
    print(f"bp_rows={result['counts']['bp']}")
    print(f"cc_rows={result['counts']['cc']}")
    print(f"missing_rows={result['missing_rows']}")

    if result['missing_counts']:
        print("missing_go_terms=")
        for go_id, count in sorted(result['missing_counts'].items()):
            print(f"  {go_id}: {count}")
    else:
        print("missing_go_terms=none")

    print(f"mf_path={result['output_paths']['molecular_function']}")
    print(f"bp_path={result['output_paths']['biological_process']}")
    print(f"cc_path={result['output_paths']['cellular_component']}")


In [ ]:
def normalize_category(go_term_category: str) -> str:
    key = go_term_category.strip().lower()
    if key not in CATEGORY_ALIASES:
        raise ValueError(f"Unsupported category: {go_term_category}")
    return CATEGORY_ALIASES[key]


def extract_go_ids(text: str) -> set[str]:
    return set(GO_PATTERN.findall(text or ""))


def normalize_protein_id(protein_id: str) -> str:
    parts = protein_id.split("|")
    if len(parts) >= 3:
        return parts[1]
    return protein_id


def build_ancestor_lookup(primary_by_go: dict[str, str], parents_by_go: dict[str, list[str]]):
    cache: dict[str, set[str]] = {}

    def ancestors(go_id: str) -> set[str]:
        canonical_go = primary_by_go.get(go_id, go_id)
        if canonical_go in cache:
            return cache[canonical_go]
        expanded = {canonical_go}
        for parent_go in parents_by_go.get(canonical_go, []):
            expanded.update(ancestors(parent_go))
        cache[canonical_go] = expanded
        return expanded

    return ancestors


def expand_terms_for_category(
    go_ids: set[str],
    category: str,
    namespace_by_go: dict[str, str],
    primary_by_go: dict[str, str],
    ancestors,
) -> set[str]:
    wanted_namespace = CATEGORY_TO_NAMESPACE[category]
    root_go = ROOT_GO_BY_CATEGORY[category]
    expanded_terms: set[str] = set()
    for go_id in go_ids:
        canonical_go = primary_by_go.get(go_id, go_id)
        for ancestor_go in ancestors(canonical_go):
            if namespace_by_go.get(ancestor_go) == wanted_namespace and ancestor_go != root_go:
                expanded_terms.add(ancestor_go)
    return expanded_terms


def load_ground_truth(
    category: str,
    namespace_by_go: dict[str, str],
    primary_by_go: dict[str, str],
    ancestors,
) -> dict[str, set[str]]:
    truth_by_protein: dict[str, set[str]] = {}
    column_name = CATEGORY_TO_TEST_COLUMN[category]
    with TEST_TSV_PATH.open(newline="") as handle:
        reader = csv.DictReader(handle, delimiter="\t")
        for row in reader:
            protein_id = row["Entry"]
            go_ids = extract_go_ids(row.get(column_name, ""))
            truth_by_protein[protein_id] = expand_terms_for_category(
                go_ids,
                category,
                namespace_by_go,
                primary_by_go,
                ancestors,
            )
    return truth_by_protein


def load_predictions(
    path_to_tsv: str,
    category: str,
    namespace_by_go: dict[str, str],
    primary_by_go: dict[str, str],
    ancestors,
    allowed_proteins: set[str],
) -> dict[str, dict[str, float]]:
    predictions: dict[str, dict[str, float]] = {protein_id: {} for protein_id in allowed_proteins}
    wanted_namespace = CATEGORY_TO_NAMESPACE[category]
    with Path(path_to_tsv).open() as handle:
        for line_no, raw_line in enumerate(handle, start=1):
            line = raw_line.rstrip("\n")
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) < 3:
                raise ValueError(f"{path_to_tsv}:{line_no} does not have 3 columns")
            raw_protein_id, go_id, score_text = parts[:3]
            protein_id = normalize_protein_id(raw_protein_id)
            if protein_id not in allowed_proteins:
                continue
            canonical_go = primary_by_go.get(go_id, go_id)
            namespace = namespace_by_go.get(go_id) or namespace_by_go.get(canonical_go)
            if namespace != wanted_namespace:
                continue
            score = float(score_text)
            expanded_terms = expand_terms_for_category(
                {canonical_go},
                category,
                namespace_by_go,
                primary_by_go,
                ancestors,
            )
            for expanded_go in expanded_terms:
                old_score = predictions[protein_id].get(expanded_go)
                if old_score is None or score > old_score:
                    predictions[protein_id][expanded_go] = score
    return predictions


def compute_ic_by_term(truth_by_protein: dict[str, set[str]], universe_terms: set[str]) -> dict[str, float]:
    protein_count = len(truth_by_protein)
    term_frequency = {term: 0 for term in universe_terms}
    for true_terms in truth_by_protein.values():
        for term in true_terms:
            term_frequency[term] = term_frequency.get(term, 0) + 1
    ic_by_term = {}
    for term in universe_terms:
        frequency = term_frequency.get(term, 0)
        if frequency == 0:
            ic_by_term[term] = 0.0
        else:
            ic_by_term[term] = -math.log2(frequency / protein_count)
    return ic_by_term


def compute_pair_metrics(
    truth_by_protein: dict[str, set[str]],
    predictions_by_protein: dict[str, dict[str, float]],
) -> dict[str, float]:
    proteins = sorted(truth_by_protein)
    universe_terms: set[str] = set()
    for protein_id in proteins:
        universe_terms.update(truth_by_protein[protein_id])
        universe_terms.update(predictions_by_protein.get(protein_id, {}).keys())

    if not universe_terms:
        return {
            "fmax": 0.0,
            "smin": 0.0,
            "aupr": 0.0,
            "auc": 0.0,
            "proteins": float(len(proteins)),
            "terms": 0.0,
            "total_pairs": 0.0,
            "explicit_predictions": 0.0,
        }

    total_pairs = len(proteins) * len(universe_terms)
    total_pos = 0
    explicit_items: list[tuple[float, int]] = []
    explicit_positive_count = 0

    for protein_id in proteins:
        true_terms = truth_by_protein[protein_id]
        total_pos += len(true_terms)
        for term, score in predictions_by_protein.get(protein_id, {}).items():
            label = 1 if term in true_terms else 0
            explicit_items.append((score, label))
            explicit_positive_count += label

    explicit_predictions = len(explicit_items)
    zero_pairs = total_pairs - explicit_predictions
    zero_pos = total_pos - explicit_positive_count
    total_neg = total_pairs - total_pos
    explicit_items.sort(key=lambda item: item[0], reverse=True)

    tp = 0
    fp = 0
    best_fmax = 0.0
    prev_recall = 0.0
    aupr = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    auc = 0.0

    ic_by_term = compute_ic_by_term(truth_by_protein, universe_terms)
    total_ic = sum(ic_by_term.values())
    ru_total = sum(sum(ic_by_term[term] for term in truth_by_protein[protein_id]) for protein_id in proteins)
    mi_total = 0.0
    all_nontruth_ic_total = sum(total_ic - sum(ic_by_term[term] for term in truth_by_protein[protein_id]) for protein_id in proteins)
    best_smin = math.sqrt((ru_total / len(proteins)) ** 2 + (mi_total / len(proteins)) ** 2)

    predicted_terms_by_protein = {protein_id: set() for protein_id in proteins}
    prediction_events: list[tuple[float, str, str, bool]] = []
    for protein_id in proteins:
        true_terms = truth_by_protein[protein_id]
        for term, score in predictions_by_protein.get(protein_id, {}).items():
            prediction_events.append((score, protein_id, term, term in true_terms))
    prediction_events.sort(key=lambda item: item[0], reverse=True)

    index = 0
    while index < len(explicit_items):
        score = explicit_items[index][0]
        group_tp = 0
        group_fp = 0
        while index < len(explicit_items) and explicit_items[index][0] == score:
            group_tp += explicit_items[index][1]
            group_fp += 1 - explicit_items[index][1]
            index += 1
        tp += group_tp
        fp += group_fp
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / total_pos if total_pos else 0.0
        fmax = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
        best_fmax = max(best_fmax, fmax)
        aupr += (recall - prev_recall) * precision
        prev_recall = recall
        fpr = fp / total_neg if total_neg else 0.0
        tpr = recall
        auc += (fpr - prev_fpr) * (tpr + prev_tpr) / 2
        prev_fpr = fpr
        prev_tpr = tpr

    event_index = 0
    while event_index < len(prediction_events):
        score = prediction_events[event_index][0]
        while event_index < len(prediction_events) and prediction_events[event_index][0] == score:
            _, protein_id, term, is_true = prediction_events[event_index]
            if term not in predicted_terms_by_protein[protein_id]:
                predicted_terms_by_protein[protein_id].add(term)
                if is_true:
                    ru_total -= ic_by_term[term]
                else:
                    mi_total += ic_by_term[term]
            event_index += 1
        s_value = math.sqrt((ru_total / len(proteins)) ** 2 + (mi_total / len(proteins)) ** 2)
        best_smin = min(best_smin, s_value)

    if zero_pairs > 0:
        tp = total_pos
        fp = total_neg
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = 1.0 if total_pos else 0.0
        fmax = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
        best_fmax = max(best_fmax, fmax)
        aupr += (recall - prev_recall) * precision
        fpr = 1.0 if total_neg else 0.0
        tpr = recall
        auc += (fpr - prev_fpr) * (tpr + prev_tpr) / 2
        best_smin = min(best_smin, all_nontruth_ic_total / len(proteins))

    return {
        "fmax": best_fmax,
        "smin": best_smin,
        "aupr": aupr,
        "auc": auc,
        "proteins": float(len(proteins)),
        "terms": float(len(universe_terms)),
        "total_pairs": float(total_pairs),
        "explicit_predictions": float(explicit_predictions),
    }


def evaluate(path_to_file: str, go_term_category: str) -> dict[str, float]:
    category = normalize_category(go_term_category)
    namespace_by_go, primary_by_go, parents_by_go = load_go_metadata(OBO_PATH)
    ancestors = build_ancestor_lookup(primary_by_go, parents_by_go)
    truth_by_protein = load_ground_truth(category, namespace_by_go, primary_by_go, ancestors)
    predictions_by_protein = load_predictions(
        path_to_file,
        category,
        namespace_by_go,
        primary_by_go,
        ancestors,
        set(truth_by_protein),
    )
    metrics = compute_pair_metrics(truth_by_protein, predictions_by_protein)

    print(f"prediction_file={path_to_file}")
    print(f"truth_file={TEST_TSV_PATH}")
    print(f"category={category}")
    print(f"proteins={int(metrics['proteins'])}")
    print(f"terms={int(metrics['terms'])}")
    print(f"total_pairs={int(metrics['total_pairs'])}")
    print(f"explicit_predictions={int(metrics['explicit_predictions'])}")
    print(f"fmax={metrics['fmax']:.6f}")
    print(f"smin={metrics['smin']:.6f}")
    print(f"aupr={metrics['aupr']:.6f}")
    print(f"auc={metrics['auc']:.6f}")
    return metrics


In [ ]:
# Split examples
# main("/Users/patj/Desktop/pe3/paper/baselines/comparison/data/pe3.tsv")
# main("/Users/patj/Desktop/pe3/paper/baselines/comparison/data/sprof.tsv")

# Evaluation examples
# evaluate("/Users/patj/Desktop/pe3/paper/baselines/comparison/data/pe3.tsv", "bp")
# evaluate("/Users/patj/Desktop/pe3/paper/baselines/comparison/data/sprof.tsv", "mm")
